In [7]:
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# =========================================================================
# Configuration
# =========================================================================

PATH        = r'C:\Users\lidon\Desktop\2026spring\IDX_MLS_Analytics'
SOLD_CSV    = os.path.join(PATH, 'all_sold_residential.csv')
LISTING_CSV = os.path.join(PATH, 'all_listings_residential.csv')
DTYPE_SPEC  = {'PostalCode': str, 'ListingKey': str}

# =========================================================================
# Load datasets
# Use .copy() so the original loaded data is never modified in place
# =========================================================================

print("Loading datasets...")
sold_raw     = pd.read_csv(SOLD_CSV,    dtype=DTYPE_SPEC, low_memory=False)
listings_raw = pd.read_csv(LISTING_CSV, dtype=DTYPE_SPEC, low_memory=False)

sold     = sold_raw.copy()
listings = listings_raw.copy()

print(f"  all_sold     : {len(sold):,} rows  |  {sold.shape[1]} columns")
print(f"  all_listings : {len(listings):,} rows  |  {listings.shape[1]} columns\n")

# =========================================================================
# STEP 1 — Convert date columns to datetime
# =========================================================================

print("=" * 65)
print("STEP 1 — DATE FORMAT CONVERSION")
print("=" * 65)

DATE_COLS = [
    'CloseDate',
    'PurchaseContractDate',
    'ListingContractDate',
    'ContractStatusChangeDate',
]

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}]")
    for col in DATE_COLS:
        if col in df.columns:
            before_nulls = df[col].isna().sum()
            df[col] = pd.to_datetime(df[col], errors='coerce')
            after_nulls  = df[col].isna().sum()
            new_nulls    = after_nulls - before_nulls
            print(f"    {col:<35} converted  "
                  f"(new NaT from unparseable strings: {new_nulls:,})")
        else:
            print(f"    {col:<35} not found — skipped")

print(f"\n  Date column dtypes confirmed (sold):")
for col in DATE_COLS:
    if col in sold.columns:
        print(f"    {col:<35} {sold[col].dtype}")

# =========================================================================
# STEP 2 — Convert boolean columns
# All 7 YN columns converted using .astype("boolean").
# pandas "boolean" dtype supports NA values (unlike numpy bool).
# WaterfrontYN and BasementYN are NOT dropped here — they will appear
# in Step 4 missing value analysis (~99% and ~98% missing) and be
# dropped there together with other high-missing columns.
# =========================================================================

print("\n" + "=" * 65)
print("STEP 2 — BOOLEAN FORMAT CONVERSION (YN COLUMNS)")
print("=" * 65)

boolean_columns = [
    "AttachedGarageYN",
    "ViewYN",
    "PoolPrivateYN",
    "NewConstructionYN",
    "FireplaceYN",
    "WaterfrontYN",
    "BasementYN",
]

# Convert them into boolean
for col in boolean_columns:
    if col in sold.columns:
        sold[col] = sold[col].astype("boolean")

for col in boolean_columns:
    if col in listings.columns:
        listings[col] = listings[col].astype("boolean")

# Print dtype confirmation
print("Sold boolean column dtypes:")
print(sold[[col for col in boolean_columns if col in sold.columns]].dtypes)

print("\nList boolean column dtypes:")
print(listings[[col for col in boolean_columns if col in listings.columns]].dtypes)

# =========================================================================
# STEP 3 — Remove duplicate suffix columns (.1/.2/.3) in listings
# =========================================================================

print("\n" + "=" * 65)
print("STEP 3 — REMOVE DUPLICATE SUFFIX COLUMNS")
print("=" * 65)

def drop_duplicate_suffix_cols(df, label):
    dup_cols = [c for c in df.columns
                if c.endswith('.1') or c.endswith('.2') or c.endswith('.3')]
    if not dup_cols:
        print(f"\n  [{label}] No duplicate-suffix columns found — clean.")
        return df

    print(f"\n  [{label}] Found {len(dup_cols)} duplicate-suffix columns:")
    cols_to_drop = []
    for c in dup_cols:
        base_col = c.rsplit('.', 1)[0]
        if base_col in df.columns:
            is_identical = df[base_col].equals(df[c])
            if is_identical:
                print(f"    • {c}  identical to  {base_col}  → removed")
                cols_to_drop.append(c)
            else:
                print(f"    ⚠️  {c}  differs from  {base_col}  → kept")
        else:
            print(f"    ⚠️  {c}  no base column found → kept")
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f"  Removed {len(cols_to_drop)} columns. "
              f"Now has {df.shape[1]} columns.")
    return df

sold     = drop_duplicate_suffix_cols(sold,     'sold')
listings = drop_duplicate_suffix_cols(listings, 'listings')

# =========================================================================
# STEP 4 — Handle missing values
# 1. Drop columns with > 90% missing
#    (WaterfrontYN ~99.9% and BasementYN ~98% will be dropped here)
# 2. Keep columns with 50–90% missing if useful for analysis
# 3. Find rows in Sold with missing ClosePrice
# =========================================================================

print("\n" + "=" * 65)
print("STEP 4 — MISSING VALUE HANDLING")
print("=" * 65)

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    total = len(df)
    missing_pct = (df.isnull().sum() / total * 100).sort_values(ascending=False)

    # 4a. Columns > 90% missing → drop
    drop_cols = missing_pct[missing_pct > 90].index.tolist()
    print(f"\n  [{label}] Columns > 90% missing → DROP ({len(drop_cols)} cols):")
    for col in drop_cols:
        print(f"    • {col:<40} {missing_pct[col]:.1f}%")
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)
        print(f"  Dropped. Now has {df.shape[1]} columns.")

    # 4b. Columns 50–90% missing → report only, keep for now
    mid_cols = missing_pct[
        (missing_pct > 50) & (missing_pct <= 90)
    ]
    mid_cols = mid_cols[mid_cols.index.isin(df.columns)]
    print(f"\n  [{label}] Columns 50–90% missing → KEEP (review if useful):")
    if len(mid_cols) > 0:
        for col, pct in mid_cols.items():
            print(f"    • {col:<40} {pct:.1f}%")
    else:
        print("    None found.")

    if label == 'sold':
        sold = df
    else:
        listings = df

# 4c. Find rows in Sold with missing ClosePrice
print(f"\n  [sold] Rows with missing ClosePrice:")
missing_close = sold[sold['ClosePrice'].isna()]
print(f"  Count: {len(missing_close)}")
if len(missing_close) > 0:
    show_cols = ['ListingKey', 'CloseDate', 'ListPrice',
                 'ClosePrice', 'CountyOrParish']
    show_cols = [c for c in show_cols if c in missing_close.columns]
    print(missing_close[show_cols].to_string(index=True))

# =========================================================================
# STEP 5 — Ensure numeric fields are properly typed (float64 / int64)
# =========================================================================

print("\n" + "=" * 65)
print("STEP 5 — NUMERIC FIELD TYPE CHECK AND CONVERSION")
print("=" * 65)

NUMERIC_COLS = [
    'ClosePrice', 'ListPrice', 'OriginalListPrice',
    'LivingArea', 'LotSizeAcres', 'LotSizeSquareFeet',
    'BedroomsTotal', 'BathroomsTotalInteger',
    'DaysOnMarket', 'YearBuilt',
    'AssociationFee', 'TaxAnnualAmount',
    'GarageSpaces', 'ParkingTotal',
    'Latitude', 'Longitude',
]

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}]")
    for col in NUMERIC_COLS:
        if col not in df.columns:
            continue
        current_dtype = str(df[col].dtype)
        if current_dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='coerce')
            print(f"    {col:<35} object → float64  (converted)")
        else:
            print(f"    {col:<35} {current_dtype:<10} ✅")

# =========================================================================
# STEP 6 — Geographic flags
# Flag 3 boolean columns: lat_0, long_0, out_of_state
# CA range: Longitude (-125 to -114), Latitude (32 to 42)
# =========================================================================

print("\n" + "=" * 65)
print("STEP 6 — GEOGRAPHIC FLAGS")
print("=" * 65)

CA_LON_MIN, CA_LON_MAX = -125, -114
CA_LAT_MIN, CA_LAT_MAX =   32,   42

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}]")

    if 'Latitude' not in df.columns or 'Longitude' not in df.columns:
        print("    Latitude/Longitude not found — skipping")
        continue

    df['lat_0'] = df['Latitude'].notna() & (df['Latitude'] == 0)
    print(f"    lat_0          (Latitude == 0)          : "
          f"{df['lat_0'].sum():,}")

    df['long_0'] = df['Longitude'].notna() & (df['Longitude'] == 0)
    print(f"    long_0         (Longitude == 0)         : "
          f"{df['long_0'].sum():,}")

    df['out_of_state'] = (
        df['Latitude'].notna() & df['Longitude'].notna() &
        ~df['lat_0'] & ~df['long_0'] &
        (
            (df['Longitude'] < CA_LON_MIN) |
            (df['Longitude'] > CA_LON_MAX) |
            (df['Latitude']  < CA_LAT_MIN) |
            (df['Latitude']  > CA_LAT_MAX)
        )
    )
    print(f"    out_of_state   (outside CA bbox)        : "
          f"{df['out_of_state'].sum():,}")

    total     = len(df)
    any_geo   = df['lat_0'] | df['long_0'] | df['out_of_state']
    clean_geo = (~any_geo & df['Latitude'].notna()).sum()
    print(f"    Clean coordinates : {clean_geo:,} / {total:,} "
          f"({clean_geo/total*100:.1f}%)")

    if label == 'sold':
        sold = df
    else:
        listings = df

# =========================================================================
# STEP 7 — Remove invalid numeric values
# =========================================================================

print("\n" + "=" * 65)
print("STEP 7 — REMOVE INVALID NUMERIC VALUES")
print("=" * 65)

print("\n  [sold]")
if 'ClosePrice' in sold.columns:
    invalid = (sold['ClosePrice'].notna() & (sold['ClosePrice'] <= 0)).sum()
    sold = sold[~(sold['ClosePrice'].notna() & (sold['ClosePrice'] <= 0))]
    print(f"    ClosePrice <= 0  removed : {invalid:,}")
if 'ListPrice' in sold.columns:
    count_lp = (sold['ListPrice'].notna() & (sold['ListPrice'] <= 0)).sum()
    print(f"    ListPrice <= 0   count   : {count_lp:,}  "
          f"(kept for now — pending discussion)")

print("\n  [listings]")
if 'ListPrice' in listings.columns:
    invalid = (listings['ListPrice'].notna() &
               (listings['ListPrice'] <= 0)).sum()
    listings = listings[
        ~(listings['ListPrice'].notna() & (listings['ListPrice'] <= 0))
    ]
    print(f"    ListPrice <= 0   removed : {invalid:,}")
print(f"    ClosePrice               skipped (ignore in List dataset for now)")

BOTH_RULES = [
    ('LivingArea',            '<=', 0),
    ('BedroomsTotal',         '<',  0),
    ('BathroomsTotalInteger', '<',  0),
    ('DaysOnMarket',          '<',  0),
]

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}] — shared rules")
    for col, op, threshold in BOTH_RULES:
        if col not in df.columns:
            continue
        before_rule = len(df)
        if op == '<=':
            mask = df[col].notna() & (df[col] <= threshold)
        else:
            mask = df[col].notna() & (df[col] < threshold)
        df = df[~mask]
        removed = before_rule - len(df)
        print(f"    {col:<28} {op} {threshold}  removed: {removed:,}")
    if label == 'sold':
        sold = df
    else:
        listings = df

print(f"\n  Rows after Step 7:")
print(f"    sold     : {len(sold):,}")
print(f"    listings : {len(listings):,}")

# =========================================================================
# STEP 8 — Duplicate row check
# =========================================================================

print("\n" + "=" * 65)
print("STEP 8 — DUPLICATE ROW CHECK")
print("=" * 65)

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}]")
    before = len(df)
    if 'ListingKey' in df.columns:
        key_dups = df.duplicated(subset=['ListingKey']).sum()
        print(f"    Rows with duplicate ListingKey : {key_dups:,}")
        df = df.drop_duplicates(subset=['ListingKey'], keep='first')
        print(f"    Removed                        : {before - len(df):,}")
        print(f"    Rows remaining                 : {len(df):,}")
    else:
        print("    ListingKey not found — skipping")
    if label == 'sold':
        sold = df.reset_index(drop=True)
    else:
        listings = df.reset_index(drop=True)

# =========================================================================
# STEP 9 — Date consistency flags (flag only, not remove)
# =========================================================================

print("\n" + "=" * 65)
print("STEP 9 — DATE CONSISTENCY FLAGS (flag only, not remove)")
print("=" * 65)

for df, label in [(sold, 'sold'), (listings, 'listings')]:
    print(f"\n  [{label}]")

    if 'ListingContractDate' in df.columns and 'CloseDate' in df.columns:
        mask = df['ListingContractDate'].notna() & df['CloseDate'].notna()
        df['listing_after_close_flag'] = (
            mask & (df['ListingContractDate'] > df['CloseDate'])
        )
        print(f"    listing_after_close_flag  : "
              f"{df['listing_after_close_flag'].sum():,}")
    else:
        df['listing_after_close_flag'] = False

    if 'PurchaseContractDate' in df.columns and 'CloseDate' in df.columns:
        mask = df['PurchaseContractDate'].notna() & df['CloseDate'].notna()
        df['purchase_after_close_flag'] = (
            mask & (df['PurchaseContractDate'] > df['CloseDate'])
        )
        print(f"    purchase_after_close_flag : "
              f"{df['purchase_after_close_flag'].sum():,}")
    else:
        df['purchase_after_close_flag'] = False

    df['negative_timeline_flag'] = (
        df['listing_after_close_flag'] | df['purchase_after_close_flag']
    )
    print(f"    negative_timeline_flag    : "
          f"{df['negative_timeline_flag'].sum():,}  (union of above two)")

    if label == 'sold':
        sold = df
    else:
        listings = df

# =========================================================================
# Save outputs
# =========================================================================

print("\n" + "=" * 65)
print("SAVING CLEANED DATASETS")
print("=" * 65)

sold_out     = os.path.join(PATH, 'all_sold_cleaned.csv')
listings_out = os.path.join(PATH, 'all_listings_cleaned.csv')

sold.to_csv(sold_out,         index=False, encoding='utf-8')
listings.to_csv(listings_out, index=False, encoding='utf-8')

print(f"  Saved: all_sold_cleaned.csv     — "
      f"{len(sold):,} rows  |  {sold.shape[1]} columns")
print(f"  Saved: all_listings_cleaned.csv — "
      f"{len(listings):,} rows  |  {listings.shape[1]} columns")

# =========================================================================
# Final summary
# =========================================================================

print("\n" + "=" * 65)
print("WEEK 4-5 CLEANING SUMMARY")
print("=" * 65)
print(f"  sold     rows   : {len(sold_raw):,} → {len(sold):,}")
print(f"  listings rows   : {len(listings_raw):,} → {len(listings):,}")
print(f"  sold     columns: {sold_raw.shape[1]} → {sold.shape[1]}")
print(f"  listings columns: {listings_raw.shape[1]} → {listings.shape[1]}")

Loading datasets...
  all_sold     : 448,033 rows  |  84 columns
  all_listings : 615,739 rows  |  84 columns

STEP 1 — DATE FORMAT CONVERSION

  [sold]
    CloseDate                           converted  (new NaT from unparseable strings: 0)
    PurchaseContractDate                converted  (new NaT from unparseable strings: 0)
    ListingContractDate                 converted  (new NaT from unparseable strings: 0)
    ContractStatusChangeDate            converted  (new NaT from unparseable strings: 0)

  [listings]
    CloseDate                           converted  (new NaT from unparseable strings: 0)
    PurchaseContractDate                converted  (new NaT from unparseable strings: 0)
    ListingContractDate                 converted  (new NaT from unparseable strings: 0)
    ContractStatusChangeDate            converted  (new NaT from unparseable strings: 0)

  Date column dtypes confirmed (sold):
    CloseDate                           datetime64[ns]
    PurchaseContractDate  